In [1]:
import sys
import asyncio
import time
import os

import numpy as np

from lsst.ts import salobj

from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from lsst.ts.observatory.control.utils import RotType

In [2]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [3]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [4]:
#Start classes
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.01 sec
Read 100 history items for RemoteEvent(ATHeaderService, 0, heartbeat)
Read 100 history items for RemoteEvent(ATHeaderService, 0, largeFileObjectAvailable)
Read 100 history items for RemoteEvent(ATHeaderService, 0, logMessage)
Read 24 history items for RemoteEvent(ATHeaderService, 0, summaryState)
Read historical data in 0.05 sec
Read 100 history items for RemoteEvent(ATArchiver, 0, heartbeat)
Read 1 history items for RemoteEvent(ATArchiver, 0, imageInOODS)
Read 1 history items for RemoteEvent(ATArchiver, 0, imageRetrievalForArchiving)
Read 12 history items for RemoteEvent(ATArchiver, 0, summ

[[None, None, None, None, None, None, None], [None, None, None, None]]

mountPositions DDS read queue is full (100 elements); data may be lost


In [16]:
# enable components
# already enabled
await atcs.enable({"atdome": "current", "ataos": "current", "athexapod": "current"})
#await latiss.enable()

Enabling all components
Gathering settings.
Received settings from users.: {'atdome': 'current', 'ataos': 'current', 'athexapod': 'current'}
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for atpneumatics.
Complete settings for atdometrajectory.
Settings versions: {'atdome': 'current', 'ataos': 'current', 'athexapod': 'current', 'atmcs': '', 'atptg': '', 'atpneumatics': '', 'atdometrajectory': ''}
[atmcs]::[<State.FAULT: 3>, <State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atptg]::[<State.FAULT: 3>, <State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[ataos]::[<State.ENABLED: 2>]
[atpneumatics]::[<State.ENABLED: 2>]
[athexapod]::[<State.ENABLED: 2>]
[atdome]::<State.ENABLED: 2>
[atdometrajectory]::[<State.ENABLED: 2>]
All components in <State.ENABLED: 2>.


In [ ]:
await atcs.slew_object('TIC 152361572', rot_type=RotType.PhysicalSky, rot=55)

In [ ]:
await atcs.slew_object('HD 31811', rot_type=RotType.PhysicalSky, rot=45)

In [ ]:
await atcs.slew_object('HD 90105', rot=-110.0, rot_type=RotType.PhysicalSky)

In [18]:
await atcs.point_azel(180.0, 80.0, rot_tel=-90.0)

Sending command
Stop tracking.
Tracking state: <AtMountState.TRACKINGENABLED: 9>
Tracking state: <AtMountState.STOPPING: 10>
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.003 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = +070.000 deg 
[Telescope] delta Alt = -000.002 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = +068.625 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = +064.677 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = +058.670 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +052.676 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +046.

In [ ]:
await latiss.take_bias(1)

In [ ]:
# Take 50 biases seq # 174-223
# Added wait to stop killing the recent images
for i in range(50):
    await asyncio.sleep(2.0)
    await latiss.take_bias(1)

In [ ]:
# Take 10 10 second darks 224-233
await latiss.take_darks(10.0, 10)

In [ ]:
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [ ]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.ENABLED, settingsToApply='current')

In [ ]:
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.ENABLED, settingsToApply='current')

In [ ]:
await atcs.prepare_for_flatfield()

In [ ]:
# Take 10 2 second flats 234-243
await latiss.take_flats(2.0, 10, filter='RG610', grating='empty_1')

In [ ]:
# Take flats for PTC 244-283
# Added wait to stop killing the recent images
for i in range(20):
    exp = 0.2 * float(i+1)
    await latiss.take_flats(exp, 1, filter='empty_1', grating='empty_1')
    if exp < 2.0:
        await asyncio.sleep(2.0)
    await latiss.take_flats(exp, 1, filter='empty_1', grating='empty_1')


In [ ]:
await atcs.prepare_for_onsky()

In [ ]:
# Opens/Closes dropout shutter
await atcs.rem.atdome.cmd_moveShutterDropoutDoor.set_start(open=False)

In [ ]:
# Don't forget the 'settingsToApply='current' or it won't work!!
#await salobj.set_summary_state(atcs.rem.ataos, salobj.State.ENABLED, settingsToApply='current')

In [ ]:
await atcs.rem.ataos.cmd_enableCorrection.set_start(
    m1=True, hexapod=True, atspectrograph=True, timeout=atcs.long_timeout
)

In [ ]:
# HD72514 Az~120, Alt~65 m=8.05

In [ ]:
current_az = 120.0
current_el = 65.0
current_rot = -50.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

In [ ]:
await salobj.set_summary_state(atcs.rem.atptg, salobj.State.ENABLED)

In [ ]:
await salobj.set_summary_state(atcs.rem.atmcs, salobj.State.ENABLED)

In [ ]:
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.ENABLED)

In [ ]:
#await latiss.setup_instrument()

In [ ]:
#await latiss.setup_instrument

In [ ]:
await latiss.setup_atspec(filter='RG610')

In [ ]:
# HD55070 m = 5.45, Az = +85, Alt = +74
# HD78876 m = 7.2, Az = +90, Alt = +60
# HD72514 m = 7.6
await atcs.slew_object('HD 90105', rot=-110.0, rot_type=RotType.PhysicalSky)

In [ ]:
await latiss.take_object(5.0, 1, filter='RG610', grating='empty_1')

In [ ]:
await atcs.add_point_data()

In [ ]:
#await latiss.setup_atspec(filter='RG610', grating='empty_1')

In [ ]:
# Take a set of images on both sides of current best focus
# ran with -0.40 and 0.05, which gave twice as many as needed

starting_z_offset = -0.20
z_offset_increment = 0.05
nsteps = int((-2 * starting_z_offset) / z_offset_increment) + 1
total_z_offset = 0.0
await atcs.rem.ataos.cmd_offset.set_start(z=starting_z_offset)
total_z_offset += starting_z_offset
print(f"Total z offset = {total_z_offset}")
await asyncio.sleep(2)
for i in range(nsteps):
    await latiss.take_object(5.0, 1, filter='RG610', grating='empty_1')
    await atcs.rem.ataos.cmd_offset.set_start(z=z_offset_increment)
    total_z_offset += z_offset_increment
    print(f"Total z offset = {total_z_offset}")
    
# Put offset back where it was
await atcs.rem.ataos.cmd_offset.set_start(z=-total_z_offset)
total_z_offset -= total_z_offset
print(f"Total z offset = {total_z_offset}")
    

In [ ]:
await atcs.rem.ataos.cmd_offset.set_start(z=-0.095)

In [ ]:
atcs.offset_xy?

In [ ]:
# Move the star to the bottom of the field
await atcs.offset_xy(y=140, x=0)

In [ ]:
# Currently at 2224, 162
# Want 1750, 300
# Shift x=47, y=135
await atcs.offset_xy(y=126, x=55)

In [ ]:
# Take a set of images on both sides of current best focus
# ran with -0.40 and 0.05, which gave twice as many as needed

starting_z_offset = -0.30
z_offset_increment = 0.03
nsteps = int((-2 * starting_z_offset) / z_offset_increment) + 1
total_z_offset = 0.0
await atcs.rem.ataos.cmd_offset.set_start(z=starting_z_offset)
total_z_offset += starting_z_offset
print(f"Total z offset = {total_z_offset}")
await asyncio.sleep(2)
for i in range(nsteps):
    await latiss.take_object(25.0, 1, filter='RG610', grating='holo4_003')
    await atcs.rem.ataos.cmd_offset.set_start(z=z_offset_increment)
    total_z_offset += z_offset_increment
    print(f"Total z offset = {total_z_offset}")
    
# Put offset back where it was
await atcs.rem.ataos.cmd_offset.set_start(z=-total_z_offset)
total_z_offset -= total_z_offset
print(f"Total z offset = {total_z_offset}")
    

In [ ]:
# To reset all offsets
tmp = await atcs.rem.ataos.cmd_resetOffset.set_start(axis='y')

In [ ]:
atcs.rem.ataos.cmd_disableCorrection.set_start?

In [ ]:
# await atcs.slew_object('HD72514', rot=-50.0, rot_type=RotType.PhysicalSky)

In [ ]:
#await latiss.setup_atspec(filter='RG610', grating='empty_1')

In [ ]:
await latiss.take_object(5.0, 1, filter='RG610', grating='holo4_003')

In [ ]:
await atcs.rem.ataos.cmd_disableCorrection.set_start(atspectrograph=True)

In [ ]:
await atcs.rem.ataos.cmd_enableCorrection.set_start(atspectrograph=True)

In [ ]:
# ronch190lpm, ronchi170lpm, holo4_003
await latiss.setup_atspec(filter='RG610', grating='holo4_003')

In [ ]:
# ronch190lpmm, ronchi170lpmm, holo4_003
await latiss.setup_atspec(filter='RG610', grating='holo4_003')

In [ ]:
await latiss.take_object(5.0, 1, filter='RG610', grating='holo4_003')

In [ ]:
await latiss.take_object(1.0, 1, filter='empty_1', grating='empty_1')

In [19]:
await atcs.shutdown()

Disabling ATAOS corrections
Disabling ATAOS corrections.
Closing M1 cover vent gates.
Cover state <MirrorCoverState.CLOSED: 6>
M1 cover already closed.
M1 vent state <VentsPosition.CLOSED: 1>
M1 vents already closed.
Skipping closing dome shutter and slewing dome to park position.
Disable ATDomeTrajectory
Slew telescope to Park position.
Sending command
Stop tracking.
Tracking state: <AtMountState.TRACKINGENABLED: 9>
Tracking state: <AtMountState.STOPPING: 10>
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.003 deg; delta Az= -179.999 deg; delta N1 = +000.000 deg; delta N2 = -020.000 deg 
[Telescope] delta Alt = -000.001 deg; delta Az= -179.779 deg; delta N1 = -000.000 deg; delta N2 = -019.975 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= -176.604 deg; delta N1 = -000.000 deg; delta N2 = -

In [20]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply="test")

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [ ]:
# Putting everything back in standby.
await latiss.standby()

In [ ]:
for i in range(10):
    await latiss.take_object(0.2, 1, filter='RG610', grating='empty_1')
 

In [ ]:
await latiss.take_object(1.0, 1, filter='RG610', grating='empty_1')